# Лабораторная работа 1: Прогнозирование выживаемости пациентов с циррозом

Тут исследовательская и экспериментальная часть лабораторной работы: оценка базовой модели, подбор гиперпараметров CatBoost через Optuna, финальное обучение и генерацию файла с предсказаниями

## Импорты и пути

Тот же код проекта, что и основной интерфейс командной строки

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from catboost import CatBoostClassifier

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').exists():
    ROOT = Path('/Users/kseniazaharova/Desktop/ML1Adv')
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from ml1adv_cirrhosis.config import CLASS_NAMES
from ml1adv_cirrhosis.pipeline import (
    baseline_cross_val_log_loss,
    catboost_cross_val_log_loss,
    detect_categorical_columns,
    get_default_catboost_params,
    prepare_catboost_features,
    run_optuna_study,
    split_features_target,
)

TRAIN_PATH = ROOT / 'train.csv'
TEST_PATH = ROOT / 'test.csv'
OUTPUT_PATH = ROOT / 'data' / 'results.csv'
MODEL_PATH = ROOT / 'model' / 'catboost_model.cbm'
ROOT, TRAIN_PATH, TEST_PATH

## Загрузка данных

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
X_train, y_train = split_features_target(train_df)
train_df.head()

## Обзор датасета

In [ ]:
overview = pd.DataFrame({
    'тип_данных': train_df.dtypes.astype(str),
    'пропуски': train_df.isna().sum(),
    'доля_пропусков_проц': (train_df.isna().mean() * 100).round(2),
})
overview

In [ ]:
y_train.value_counts(normalize=True).rename('доля').to_frame()

## Базовая модель

В качестве базовой модели используется пайплайн на `RandomForest` с заполнением пропусков и one-hot-кодированием категориальных признаков

In [ ]:
baseline_score = baseline_cross_val_log_loss(X_train, y_train)
baseline_score

## Подбор CatBoost через Optuna

Поиск ниже сделан компактным, чтобы его можно было воспроизвести на локальной машине

In [ ]:
best_score, best_params, trials_summary = run_optuna_study(X_train, y_train, n_trials=12)
best_score, best_params

In [ ]:
pd.DataFrame(trials_summary).sort_values('value').head()

## Финальная оценка CatBoost

При условии, что компактный запуск Optuna пропускается, проект все равно содержит заранее подобранные параметры

In [ ]:
default_params = get_default_catboost_params()
default_cv_score = catboost_cross_val_log_loss(X_train, y_train, default_params)
default_cv_score

## Обучение финальной модели на полном обучающем наборе

In [ ]:
cat_columns = detect_categorical_columns(X_train)
prepared_train = prepare_catboost_features(X_train, cat_columns)
cat_indices = [prepared_train.columns.get_loc(col) for col in cat_columns]
final_model = CatBoostClassifier(**default_params)
final_model.fit(prepared_train, y_train, cat_features=cat_indices)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
final_model.save_model(MODEL_PATH)
MODEL_PATH

## Создание файла с предсказаниями

In [ ]:
prepared_test = prepare_catboost_features(test_df[prepared_train.columns], cat_columns)
predictions = final_model.predict_proba(prepared_test)
submission = pd.DataFrame(predictions, columns=final_model.classes_)
submission = submission.reindex(columns=CLASS_NAMES, fill_value=0.0)
submission.columns = [f'Status_{col}' for col in submission.columns]
submission.insert(0, 'id', test_df['id'].values)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)
submission.head()

## Выводы

- качество базовой модели оценивается с помощью кросс-валидации
- для CatBoost выполняется подбор гиперпараметров через Optuna, после чего модель переобучается на полном обучающем наборе
- итоговый файл с предсказаниями сохраняется в `data/results.csv`